# BISVEDA Voice Call Assistant — Exotel Voicebot Applet + Gemini Live

**AI-powered multilingual voice interface for the Bureau of Indian Standards**  
(बिसवेदा — BIS + Veda)  
Problem Statement: SIH26107 | Team: fsociety

This notebook runs the **BISVEDA real-time phone voice assistant** behind an
Exotel inbound Voicebot/App Bazaar flow. There is intentionally **no Exotel
outbound REST call in this notebook**.

```text
Your phone
   ↓
Exotel ExoPhone / BISVEDA number
   ↓
App Bazaar inbound flow
   ↓
Voicebot Applet (bidirectional AgentStream)
   ↓
wss://<ngrok>/bisveda/media?sample-rate=8000
   ↓
FastAPI WebSocket bridge
   ↓
Gemini Live (audio in/out + function calling)
   ↓
BISVEDA tools / mock backend
```

### Why this notebook differs from the Twilio version

| Concern | Twilio notebook | This Exotel notebook |
|---|---|---|
| Call initiation | Twilio REST API | **Caller dials ExoPhone** |
| Telephony control | TwiML `<Connect><Stream>` | **Exotel Voicebot Applet** |
| WebSocket route | `/bisveda/media` | `/bisveda/media` |
| Incoming audio | 8 kHz μ-law | **8 kHz raw PCM16** by default |
| Outgoing audio | PCM16 → μ-law | **PCM16 only** |
| Event field names | camelCase | **snake_case** |
| Barge-in | Twilio `clear` | **Exotel `clear`** |
| Exotel API credentials | Required for dialing | **Not needed for the prototype call path** |

Current Exotel AgentStream documentation specifies raw 16-bit signed little-endian
mono PCM, default 8 kHz, bidirectional audio for Voicebot, and `connected`,
`start`, `media`, `dtmf`, `mark`, and `stop` events. The same WebSocket accepts
`media`, `mark`, and `clear` messages from the bot. The current inbound flow
model is ExoPhone → App Bazaar → Voicebot applet → WSS bot endpoint.

Official docs:
- https://developer.exotel.com/docs/agentstream/what-to-use-when
- https://developer.exotel.com/docs/agentstream/websocket-protocol

### Voice flow
1. A user calls the Exotel number attached to the App Bazaar flow.
2. Exotel opens the `/bisveda/media` WebSocket and sends `connected` + `start`.
3. The notebook creates a Gemini Live session **only after `start`**, so the
   greeting cannot race the telephony stream setup.
4. BISVEDA greets the caller in Hindi + English and invites a question.
5. Caller audio is decoded from base64 raw PCM16 at 8 kHz and resampled to
   16 kHz PCM16 for Gemini Live.
6. Gemini audio is resampled from 24 kHz PCM16 to 8 kHz PCM16 and sent back
   to Exotel in protocol-sized chunks.
7. Gemini function calls invoke the BISVEDA backend tools and speak the result.
8. DTMF events are logged/ignored; the experience is voice-first.
9. Calls and tool usage are written incrementally to SQLite + JSONL logs.

### Important
This is a prototype. The backend functions below are mock BISVEDA services.
Replace them with your real RAG, verification APIs, QCO data, laboratory
search, complaint system, and knowledge-graph services when integrating the
production backend.


In [ ]:
# 1. Install dependencies
#
# No Twilio SDK is required. audioop-lts provides the legacy-compatible
# audioop.ratecv implementation on modern Python environments.
!pip -q install -U google-genai fastapi uvicorn pyngrok requests websockets pandas audioop-lts


In [ ]:
# 2. Configuration
#
# Loads API keys and Exotel credentials with safe environment/secret fallbacks.
# Gemini 3.1 Flash Live handles real-time audio + tool calling.

import os

def _get_secret(key, default=""):
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(key)
        if val:
            return str(val)
    except Exception:
        pass
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return str(val)
    except Exception:
        pass
    return os.environ.get(key, default)

# ── Gemini ──
GEMINI_API_KEY = _get_secret("VOICE_GEMINI_API_KEY") or _get_secret("GEMINI_API_KEY") or "AQ.Ab8RN6JrOHqgvRTeVBWciSg1P4LcTEuVq6lDEh7tF0EctC-1Mw"
GEMINI_MODEL   = _get_secret("VOICE_GEMINI_MODEL", "gemini-3.1-flash-live-preview")

# ── Exotel Telephony ──
EXOTEL_ACCOUNT_SID = _get_secret("EXOTEL_ACCOUNT_SID", "none6723")
EXOTEL_API_KEY     = _get_secret("EXOTEL_API_KEY", "90b2ea024e1f7ec958f6a3bf1d775952604fa5d65c64d90f")
EXOTEL_API_TOKEN   = _get_secret("EXOTEL_API_TOKEN", "b9e9b63bd4e658fa6eb5562a70a30433e900b2f4adc4ae23")
EXOTEL_CALLER_ID   = _get_secret("EXOTEL_CALLER_ID", "01414939968")
EXOTEL_APP_ID      = _get_secret("EXOTEL_APP_ID", "1349660")
MY_PHONE_NUMBER    = _get_secret("MY_PHONE_NUMBER", "9529512485")

# ── Server / tunnel ──
PORT               = int(_get_secret("BISVEDA_VOICE_PORT", "8001"))
NGROK_AUTHTOKEN    = _get_secret("NGROK_AUTHTOKEN", "3JuYjB10Je24Y4vDMc7nmD8yvqk_6EvETNmVk772rHuTkJVLV")
EXOTEL_SAMPLE_RATE = 8000
EXOTEL_CHANNELS    = 1
EXOTEL_BIT_DEPTH   = 16

# ── Work directory & database ──
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "."
os.makedirs(WORK_DIR, exist_ok=True)
DB_PATH        = os.path.join(WORK_DIR, "bisveda_voice.db")
LOG_FILE       = os.path.join(WORK_DIR, "bisveda_exotel_voice.log")
LOG_JSONL_FILE = os.path.join(WORK_DIR, "bisveda_exotel_voice.jsonl")

print("Configuration loaded successfully.")
print(f"Gemini model:       {GEMINI_MODEL}")
print(f"Exotel Account SID: {EXOTEL_ACCOUNT_SID}")
print(f"Exotel Caller ID:   {EXOTEL_CALLER_ID}")
print(f"My Phone Number:    {MY_PHONE_NUMBER}")
print(f"Exotel Audio:       {EXOTEL_SAMPLE_RATE} Hz / PCM16 mono")


In [ ]:
# 3. Imports + structured logging

import asyncio
import base64
import json
import logging
import sqlite3
import threading
import time
import traceback
import audioop

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from google import genai
from google.genai import types

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("bisveda-exotel-voice")

with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("BISVEDA VOICE CALL ASSISTANT — CALL LOG (EXOTEL)\n")
    f.write("=" * 80 + "\n")
open(LOG_JSONL_FILE, "w", encoding="utf-8").close()


class StructuredLogger:
    """Console + text log + JSONL logger with a consistent call/stage field."""
    def __init__(self, text_path, jsonl_path):
        self.text_path = text_path
        self.jsonl_path = jsonl_path
        self._lock = threading.Lock()

    def _write(self, level, stage, message, call_id=None, extra=None):
        ts = time.strftime("%Y-%m-%d %H:%M:%S")
        line = (
            f"[{ts}] [{level}] [{stage}]"
            + (f" [call={call_id}]" if call_id else "")
            + f" {message}"
        )
        record = {
            "ts": ts, "level": level, "stage": stage,
            "call_id": call_id, "message": message,
        }
        if extra:
            record["extra"] = extra
        with self._lock:
            print(line, flush=True)
            with open(self.text_path, "a", encoding="utf-8") as f:
                f.write(line + "\n")
            with open(self.jsonl_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    def info(self, stage, message, call_id=None, extra=None):
        self._write("INFO", stage, message, call_id, extra)

    def warn(self, stage, message, call_id=None, extra=None):
        self._write("WARN", stage, message, call_id, extra)

    def error(self, stage, message, call_id=None, extra=None):
        self._write("ERROR", stage, message, call_id, extra)


logger = StructuredLogger(LOG_FILE, LOG_JSONL_FILE)
print("Structured logger ready ->", LOG_FILE)


In [ ]:
# 4. Database layer — call_sessions + query_log
#
#   call_sessions : one row per voice call (caller info, language, duration)
#   query_log     : one row per USER QUERY within a call (what they asked,
#                   which tool was invoked, what was returned). A single call
#                   can have multiple queries (multi-turn conversation).
#
# Incremental writes: every tool call UPSERTs immediately, so a dropped call
# still retains all queries answered up to that point.

def get_db_connection():
    conn = sqlite3.connect(DB_PATH, timeout=10, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_db_connection()
    try:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS call_sessions (
                call_id           TEXT PRIMARY KEY,
                phone_number      TEXT,
                detected_language TEXT,
                call_started_at   TEXT,
                call_ended_at     TEXT,
                duration_seconds  REAL,
                total_queries     INTEGER DEFAULT 0,
                status            TEXT DEFAULT 'in_progress',
                end_reason        TEXT,
                raw_transcript    TEXT,
                satisfaction_score TEXT
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS query_log (
                query_id         INTEGER PRIMARY KEY AUTOINCREMENT,
                call_id          TEXT NOT NULL,
                timestamp        TEXT,
                user_query_text  TEXT,
                detected_intent  TEXT,
                tool_called      TEXT,
                tool_args        TEXT,
                tool_result      TEXT,
                response_summary TEXT,
                language         TEXT,
                FOREIGN KEY (call_id) REFERENCES call_sessions(call_id)
            )
        """)
        conn.commit()
        logger.info("DB_INIT", f"Database ready at {DB_PATH}")
    except Exception as e:
        logger.error("DB_ERROR", f"init_db failed: {type(e).__name__}: {e}")
        raise
    finally:
        conn.close()


def ensure_call_session(call_id, phone_number):
    """Create the call_sessions row at call start."""
    conn = get_db_connection()
    try:
        conn.execute(
            """INSERT OR IGNORE INTO call_sessions
               (call_id, phone_number, call_started_at, status)
               VALUES (?, ?, ?, 'in_progress')""",
            (call_id, phone_number, time.strftime("%Y-%m-%d %H:%M:%S")),
        )
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"ensure_call_session failed: {e}", call_id)
    finally:
        conn.close()


def update_call_session(call_id, **fields):
    """Update call session fields (language, status, etc.)."""
    allowed = {
        "detected_language", "call_ended_at", "duration_seconds",
        "total_queries", "status", "end_reason", "satisfaction_score",
    }
    clean = {k: v for k, v in fields.items() if k in allowed and v is not None}
    if not clean:
        return
    set_clause = ", ".join(f"{k} = ?" for k in clean)
    values = list(clean.values()) + [call_id]
    conn = get_db_connection()
    try:
        conn.execute(f"UPDATE call_sessions SET {set_clause} WHERE call_id = ?", values)
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"update_call_session failed: {e}", call_id)
    finally:
        conn.close()


def log_query(call_id, user_query_text, detected_intent, tool_called,
              tool_args, tool_result, response_summary, language=None):
    """Log each user query and the tool result within a call."""
    conn = get_db_connection()
    try:
        conn.execute(
            """INSERT INTO query_log
               (call_id, timestamp, user_query_text, detected_intent,
                tool_called, tool_args, tool_result, response_summary, language)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
            (
                call_id, time.strftime("%Y-%m-%d %H:%M:%S"),
                user_query_text, detected_intent, tool_called,
                json.dumps(tool_args, ensure_ascii=False) if tool_args else None,
                json.dumps(tool_result, ensure_ascii=False) if tool_result else None,
                response_summary, language,
            ),
        )
        # Increment query count
        conn.execute(
            "UPDATE call_sessions SET total_queries = total_queries + 1 WHERE call_id = ?",
            (call_id,),
        )
        conn.commit()
        logger.info("QUERY_LOGGED", f"intent={detected_intent} tool={tool_called}", call_id)
    except Exception as e:
        logger.error("DB_ERROR", f"log_query failed: {e}", call_id)
    finally:
        conn.close()


def append_transcript(call_id, speaker, text):
    conn = get_db_connection()
    try:
        row = conn.execute(
            "SELECT raw_transcript FROM call_sessions WHERE call_id = ?", (call_id,)
        ).fetchone()
        existing = (row["raw_transcript"] if row and row["raw_transcript"] else "")
        line = f"[{speaker}] {text}\n"
        conn.execute(
            "UPDATE call_sessions SET raw_transcript = ? WHERE call_id = ?",
            (existing + line, call_id),
        )
        conn.commit()
    except Exception as e:
        logger.error("DB_ERROR", f"append_transcript failed: {e}", call_id)
    finally:
        conn.close()


def fetch_all_sessions():
    conn = get_db_connection()
    try:
        rows = conn.execute(
            "SELECT * FROM call_sessions ORDER BY call_started_at DESC"
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


def fetch_all_queries():
    conn = get_db_connection()
    try:
        rows = conn.execute(
            """SELECT q.*, s.phone_number, s.detected_language
               FROM query_log q
               LEFT JOIN call_sessions s ON s.call_id = q.call_id
               ORDER BY q.timestamp DESC"""
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


init_db()
print("BISVEDA voice database initialised.")


In [ ]:
# 5. Mock BISVEDA backend services
#
# In production these would call the real BISVEDA microservices (Standards
# RAG pipeline, ISI/HUID verification APIs, QCO compliance checker, etc.).
# Here they return realistic mock data so the voice flow can be tested
# end-to-end without the full backend.
#
# Replace each function body with actual API calls when integrating.

MOCK_STANDARDS = {
    "IS 1239": {
        "number": "IS 1239",
        "title": "Mild Steel Tubes, Tubulars and Other Wrought Steel Fittings",
        "year": "2004 (Reaffirmed 2019)",
        "scope": "Covers requirements for mild steel tubes used in water, gas, and steam applications",
        "qco_status": "Under QCO — mandatory BIS certification required",
        "certification_scheme": "Scheme-I (ISI Mark)",
    },
    "IS 10500": {
        "number": "IS 10500",
        "title": "Drinking Water — Specification",
        "year": "2012 (Second Revision)",
        "scope": "Specifies requirements for drinking water quality parameters",
        "qco_status": "Referenced in QCOs for packaged drinking water (IS 14543)",
        "certification_scheme": "Scheme-I (ISI Mark) for packaged water",
    },
    "IS 616": {
        "number": "IS 616",
        "title": "Safety Requirements for Electrical Equipment",
        "year": "2017",
        "scope": "General requirements for safety of household and similar electrical appliances",
        "qco_status": "Under QCO — 48 categories of electronics under compulsory registration",
        "certification_scheme": "Scheme-VII (CRS) for electronics",
    },
    "IS 2062": {
        "number": "IS 2062",
        "title": "Hot Rolled Medium and High Tensile Structural Steel",
        "year": "2011",
        "scope": "Covers hot-rolled structural steel plates, strips, shapes and sections",
        "qco_status": "Under QCO — mandatory BIS certification",
        "certification_scheme": "Scheme-I (ISI Mark)",
    },
}

MOCK_ISI_MARKS = {
    "CM/L-1234567": {
        "valid": True,
        "manufacturer": "Demo Steel Industries Pvt. Ltd.",
        "product": "Mild Steel Tubes (IS 1239)",
        "licence_valid_until": "2027-03-31",
        "status": "Active",
    },
    "CM/L-9999999": {
        "valid": False,
        "status": "Not found in BIS records — possibly counterfeit",
    },
}

MOCK_HUIDS = {
    "AB1234CD": {
        "valid": True,
        "item_type": "Gold Ring — 22 Karat",
        "hallmarking_centre": "Mumbai AHC-127",
        "purity": "22K (916)",
        "date_hallmarked": "2026-05-14",
    },
}

MOCK_LABS = [
    {
        "name": "BIS Testing Lab — Mumbai",
        "address": "Manakalaya, E9, MIDC, Andheri East, Mumbai 400093",
        "scope": "Steel, Cement, Electrical, Food Products",
        "phone": "022-28262720",
    },
    {
        "name": "National Test House — Kolkata",
        "address": "Block CP, Sector V, Salt Lake, Kolkata 700091",
        "scope": "Metals, Chemicals, Textiles, Rubber",
        "phone": "033-23675088",
    },
    {
        "name": "STQC Lab — Delhi",
        "address": "Electronics Niketan, CGO Complex, New Delhi 110003",
        "scope": "Electronics, IT Products, Telecom Equipment",
        "phone": "011-24363083",
    },
]


def backend_search_standards(query_text, max_results=3):
    """Search BIS standards by keyword. In production: RAG pipeline query."""
    query_lower = query_text.lower()
    results = []
    for key, std in MOCK_STANDARDS.items():
        if (query_lower in key.lower()
            or query_lower in std["title"].lower()
            or query_lower in std["scope"].lower()):
            results.append(std)
    if not results:
        # Fuzzy fallback: return all if no match
        results = list(MOCK_STANDARDS.values())[:max_results]
    return results[:max_results]


def backend_verify_isi_mark(cm_l_number):
    """Verify an ISI CM/L number. In production: BIS licence API."""
    clean = cm_l_number.strip().upper().replace(" ", "")
    # Normalize formats like "CML1234567" to "CM/L-1234567"
    if clean.startswith("CML") and "/" not in clean:
        clean = "CM/L-" + clean[3:]
    elif clean.startswith("CM/L") and "-" not in clean:
        clean = clean[:4] + "-" + clean[4:]
    return MOCK_ISI_MARKS.get(clean, {
        "valid": False,
        "status": f"CM/L number '{cm_l_number}' not found in BIS records",
    })


def backend_verify_huid(huid):
    """Verify a Hallmark Unique ID. In production: BIS HUID API."""
    clean = huid.strip().upper().replace(" ", "")
    return MOCK_HUIDS.get(clean, {
        "valid": False,
        "status": f"HUID '{huid}' not found — may be invalid or not yet registered",
    })


def backend_check_qco(product_name):
    """Check if a product falls under QCO. In production: QCO database query."""
    product_lower = product_name.lower()
    qco_products = {
        "steel": {"covered": True, "standard": "IS 2062 / IS 1239",
                  "scheme": "Scheme-I (ISI Mark)",
                  "note": "All structural steel and tubes require mandatory BIS certification"},
        "cement": {"covered": True, "standard": "IS 269 / IS 8112 / IS 12269",
                   "scheme": "Scheme-I (ISI Mark)",
                   "note": "All grades of cement are under QCO"},
        "helmet": {"covered": True, "standard": "IS 4151",
                   "scheme": "Scheme-I (ISI Mark)",
                   "note": "Protective helmets for two-wheeler riders"},
        "electronics": {"covered": True, "standard": "Multiple (CRS)",
                        "scheme": "Scheme-VII (CRS Registration)",
                        "note": "48 categories under compulsory registration"},
        "toy": {"covered": True, "standard": "IS 9873 / IS 15644",
                "scheme": "Scheme-I (ISI Mark)",
                "note": "All toys for children under 14 years"},
        "water": {"covered": True, "standard": "IS 14543",
                  "scheme": "Scheme-I (ISI Mark)",
                  "note": "Packaged drinking water and mineral water"},
    }
    for keyword, info in qco_products.items():
        if keyword in product_lower:
            return info
    return {
        "covered": False,
        "note": f"'{product_name}' does not appear to be under a current QCO. "
                "However, new QCOs are notified regularly — check the latest "
                "BIS QCO orders at bis.gov.in for the most current list.",
    }


def backend_find_labs(product_or_standard, location=None):
    """Find BIS-recognized testing labs. In production: lab directory API."""
    results = MOCK_LABS
    if location:
        loc_lower = location.lower()
        filtered = [l for l in results if loc_lower in l["address"].lower()]
        if filtered:
            results = filtered
    return results[:3]


print("Mock BISVEDA backend services ready.")
print(f"  Standards in DB: {len(MOCK_STANDARDS)}")
print(f"  ISI marks:       {len(MOCK_ISI_MARKS)}")
print(f"  HUIDs:           {len(MOCK_HUIDS)}")
print(f"  Labs:            {len(MOCK_LABS)}")


In [ ]:
# 6. Audio helpers — Exotel raw PCM16 <-> Gemini Live PCM16
#
# Exotel Voicebot/AgentStream defaults to:
#   - 8 kHz
#   - mono
#   - signed 16-bit little-endian linear PCM
#   - base64 payloads
#
# Gemini Live in this prototype receives 16 kHz PCM16 and emits 24 kHz PCM16.
# We therefore resample only; NO μ-law conversion is used here.
#
# Exotel's current protocol expects outgoing audio chunks to be protocol-sized.
# We buffer resampled audio and send 3200-byte chunks (100 ms at 8 kHz PCM16)
# whenever possible.

AUDIO_STATE = {}
OUTGOING_CHUNK_BYTES = 3200


def _audio_state(call_id):
    return AUDIO_STATE.setdefault(
        call_id,
        {"in": None, "out": None, "out_buffer": bytearray()},
    )


def exotel_to_gemini_pcm16(payload_b64: str, call_id: str) -> bytes:
    """Exotel 8 kHz raw PCM16 -> Gemini 16 kHz raw PCM16."""
    raw_pcm8k = base64.b64decode(payload_b64)
    if len(raw_pcm8k) % 2:
        raw_pcm8k = raw_pcm8k[:-1]
    state = _audio_state(call_id)
    pcm16k, state["in"] = audioop.ratecv(
        raw_pcm8k, 2, EXOTEL_CHANNELS,
        EXOTEL_SAMPLE_RATE, 16000,
        state["in"],
    )
    return pcm16k


def gemini_to_exotel_chunks(pcm24k: bytes, call_id: str, flush=False):
    """Gemini 24 kHz PCM16 -> buffered 8 kHz PCM16 base64 chunks."""
    state = _audio_state(call_id)
    pcm8k, state["out"] = audioop.ratecv(
        pcm24k, 2, 1,
        24000, EXOTEL_SAMPLE_RATE,
        state["out"],
    )
    state["out_buffer"].extend(pcm8k)

    chunks = []
    while len(state["out_buffer"]) >= OUTGOING_CHUNK_BYTES:
        chunk = bytes(state["out_buffer"][:OUTGOING_CHUNK_BYTES])
        del state["out_buffer"][:OUTGOING_CHUNK_BYTES]
        chunks.append(base64.b64encode(chunk).decode("ascii"))

    if flush and state["out_buffer"]:
        # Pad the final partial packet so it still satisfies Exotel's
        # multiple-of-320 byte requirement and is at least 3200 bytes.
        remaining = bytes(state["out_buffer"])
        pad_to = max(
            OUTGOING_CHUNK_BYTES,
            ((len(remaining) + 319) // 320) * 320,
        )
        remaining += b"\x00" * (pad_to - len(remaining))
        state["out_buffer"].clear()
        chunks.append(base64.b64encode(remaining).decode("ascii"))

    return chunks


def clear_audio_state(call_id: str):
    AUDIO_STATE.pop(call_id, None)


print("Exotel PCM audio helpers ready.")


In [ ]:
# 7. Gemini tool declarations + BISVEDA system prompt
#
# The model gets NINE tools, mapping to BISVEDA's Consumer/MSME use cases:
#   search_standards       — search the BIS standards database
#   verify_isi_mark        — verify an ISI CM/L licence number
#   verify_huid            — verify a Hallmark Unique ID
#   check_qco_compliance   — check if a product is under QCO
#   find_testing_labs      — find BIS-recognized labs / AHCs
#   get_standard_passport  — get full details of a specific standard
#   report_counterfeit     — log a counterfeit / suspicious product report
#   file_complaint         — log a consumer complaint
#   end_call               — end the phone call
#
# Tool calling happens inside the Gemini Live session regardless of the telephony provider.

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# In-memory per-call state
CALLS = {}  # call_id -> {"phone": ..., "language": ..., "queries": [...]}


TOOLS = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name="search_standards",
            description=(
                "Search the BIS standards database by keywords, product name, "
                "standard number, or industry/sector. Returns matching "
                "standards with their number, title, scope, QCO status, and "
                "applicable certification scheme. Call this when the user "
                "asks about any Indian Standard, wants to know which "
                "standard applies to a product, or asks general standards "
                "questions."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "query": {
                        "type": "STRING",
                        "description": (
                            "Search query — a standard number (e.g. 'IS 1239'), "
                            "product name (e.g. 'steel tubes'), or keywords "
                            "(e.g. 'drinking water specification'). Translate "
                            "the user's query to English keywords for search."
                        ),
                    },
                },
                "required": ["query"],
            },
        ),
        types.FunctionDeclaration(
            name="verify_isi_mark",
            description=(
                "Verify whether an ISI Mark CM/L licence number is genuine "
                "and currently active. The user may say it as 'CM/L number', "
                "'ISI number', 'licence number', or spell out digits. Extract "
                "and normalize the number before calling. Returns the "
                "manufacturer name, product, licence validity, and status."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "cm_l_number": {
                        "type": "STRING",
                        "description": "The CM/L licence number, e.g. 'CM/L-1234567'",
                    },
                },
                "required": ["cm_l_number"],
            },
        ),
        types.FunctionDeclaration(
            name="verify_huid",
            description=(
                "Verify a Hallmark Unique Identification (HUID) number "
                "engraved on hallmarked gold/silver jewellery. Returns "
                "item type, purity, hallmarking centre, and date. The user "
                "may spell out the alphanumeric code."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "huid": {
                        "type": "STRING",
                        "description": "The HUID code, e.g. 'AB1234CD'",
                    },
                },
                "required": ["huid"],
            },
        ),
        types.FunctionDeclaration(
            name="check_qco_compliance",
            description=(
                "Check whether a product or product category falls under "
                "a Quality Control Order (QCO), meaning BIS certification "
                "is mandatory before it can be manufactured or imported. "
                "Returns the applicable standard, certification scheme, "
                "and whether it is currently covered."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_name": {
                        "type": "STRING",
                        "description": (
                            "Product name or category in English, e.g. "
                            "'steel pipes', 'cement', 'electronic toys'"
                        ),
                    },
                },
                "required": ["product_name"],
            },
        ),
        types.FunctionDeclaration(
            name="find_testing_labs",
            description=(
                "Find BIS-recognized testing laboratories or Assaying & "
                "Hallmarking Centres (AHCs) for a given product or standard. "
                "Optionally filter by city/state."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_or_standard": {
                        "type": "STRING",
                        "description": "Product name or standard number to find labs for",
                    },
                    "location": {
                        "type": "STRING",
                        "description": "City or state to filter by (optional)",
                    },
                },
                "required": ["product_or_standard"],
            },
        ),
        types.FunctionDeclaration(
            name="get_standard_passport",
            description=(
                "Get the full 'Standard Passport' — a complete summary of "
                "a specific Indian Standard including its number, title, "
                "year, scope, QCO status, applicable certification scheme, "
                "and related standards. Use when the user asks for details "
                "about a specific standard they already know."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "standard_number": {
                        "type": "STRING",
                        "description": "The IS number, e.g. 'IS 1239', 'IS 10500'",
                    },
                },
                "required": ["standard_number"],
            },
        ),
        types.FunctionDeclaration(
            name="report_counterfeit",
            description=(
                "Log a report of a suspected counterfeit product bearing "
                "a fake ISI mark or HUID. Captures product description, "
                "location, and any identifying numbers. Tell the user their "
                "report has been logged and will be forwarded to the "
                "appropriate BIS regional office."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_description": {
                        "type": "STRING",
                        "description": "What the product is (e.g. 'steel pipes with ISI mark')",
                    },
                    "location": {
                        "type": "STRING",
                        "description": "Where the product was found (shop, city)",
                    },
                    "suspicious_number": {
                        "type": "STRING",
                        "description": "The CM/L or HUID number on the suspect product",
                    },
                    "additional_details": {
                        "type": "STRING",
                        "description": "Any other details the caller provided",
                    },
                },
                "required": ["product_description"],
            },
        ),
        types.FunctionDeclaration(
            name="file_complaint",
            description=(
                "Log a consumer complaint about a BIS-certified product "
                "that does not meet the standard's quality. Captures the "
                "complaint details and tells the user it will be forwarded "
                "to the BIS complaint cell."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "product_description": {
                        "type": "STRING",
                        "description": "Product with the quality issue",
                    },
                    "complaint_details": {
                        "type": "STRING",
                        "description": "What the issue is — broken, substandard, unsafe, etc.",
                    },
                    "cm_l_number": {
                        "type": "STRING",
                        "description": "CM/L number on the product, if available",
                    },
                    "purchase_location": {
                        "type": "STRING",
                        "description": "Where/when the product was bought",
                    },
                },
                "required": ["product_description", "complaint_details"],
            },
        ),
        types.FunctionDeclaration(
            name="end_call",
            description=(
                "Call this ONLY after you have already spoken a polite "
                "closing goodbye out loud. This ends the phone call. "
                "Always thank the user for contacting BISVEDA before "
                "calling this."
            ),
            parameters={
                "type": "OBJECT",
                "properties": {
                    "reason": {
                        "type": "STRING",
                        "description": (
                            "'completed' | 'user_requested_end' | "
                            "'escalated_to_human' | 'no_response' | 'other'"
                        ),
                    },
                },
            },
        ),
    ])
]


def build_system_prompt() -> str:
    """Build the BISVEDA voice assistant system prompt.

    The assistant handles queries from consumers and MSMEs about BIS
    standards, ISI marks, hallmarking, QCO compliance, and more.
    It detects the user's language from their first utterance and
    continues the entire conversation in that language."""

    return """
You are "BISVEDA" (बिसवेदा), a warm, helpful, and knowledgeable AI voice
assistant for the Bureau of Indian Standards (BIS). You are speaking on a
real phone call with someone who needs help with Indian Standards,
product certification, quality marks, or BIS services.

LANGUAGE DETECTION AND HANDLING:
- When the call starts, greet the caller briefly in Hindi first, then in
  English: "Namaste! BISVEDA mein aapka swagat hai. Welcome to BISVEDA."
- Then invite them to speak: "Aap Hindi, English, ya kisi bhi bhasha mein
  apna sawaal pooch sakte hain."
- From the caller's FIRST response, detect their language.
- Continue the ENTIRE rest of the conversation in that detected language.
- If they switch languages mid-call, follow their switch.
- Supported: Hindi, English, Marathi, Tamil, Telugu, Kannada, Bengali,
  Gujarati, Malayalam, Punjabi, and code-mixed (Hinglish).
- Keep sentences SHORT and SIMPLE — many callers have low digital literacy.

YOUR CAPABILITIES (USE THE TOOLS):
1. SEARCH STANDARDS — When the user asks about any BIS standard, product
   standard, or "which standard applies to X", call search_standards with
   English keywords translated from what they said.

2. VERIFY ISI MARK — When the user wants to check an ISI mark / CM-L
   number on a product, ask them to read the number, extract it, and
   call verify_isi_mark. Tell them clearly whether it is valid or not.

3. VERIFY HUID — When the user wants to check a hallmarking HUID on their
   gold/silver jewellery, ask them to read the code, and call verify_huid.
   Report the purity, hallmarking centre, and date.

4. CHECK QCO COMPLIANCE — When the user (typically an MSME/manufacturer)
   asks whether their product needs BIS certification, or whether it is
   under a Quality Control Order, call check_qco_compliance. Explain the
   certification scheme they need to follow.

5. FIND TESTING LABS — When the user needs to find a BIS-recognized lab
   for testing, or an Assaying & Hallmarking Centre (AHC), call
   find_testing_labs with the product/standard and optionally their city.

6. STANDARD PASSPORT — When the user asks for full details about a
   specific known standard (e.g. "Tell me about IS 1239"), call
   get_standard_passport.

7. REPORT COUNTERFEIT — When the user suspects a product has a fake ISI
   mark or HUID, collect: what the product is, where they found it, and
   the number on it, then call report_counterfeit.

8. FILE COMPLAINT — When the user has a quality complaint about a
   BIS-certified product, collect: what the product is, what the issue is,
   and the CM/L number if they have it, then call file_complaint.

CONVERSATIONAL STYLE:
- Be warm, patient, and approachable — like a helpful government service
  desk, not a bureaucrat.
- Ask ONE question at a time and wait for the answer.
- Keep every turn short (1-3 sentences) — this is a phone call.
- When reading out numbers (CM/L numbers, standard numbers), spell them
  clearly digit by digit.
- When presenting search results with multiple items, summarise the top
  result first, then ask if they want to hear more.
- If the user sounds confused, repeat more simply. Never rush them.
- If the user wants to speak to a person, politely tell them this is
  an AI helpline and suggest they contact the nearest BIS regional
  office, then call end_call with reason 'escalated_to_human'.
- After answering a query, ask: "Kya aapka koi aur sawaal hai?" (or
  equivalent in their language). If they say no, thank them warmly,
  say goodbye OUT LOUD, then call end_call.

IMPORTANT RULES:
- NEVER invent or guess standard numbers, QCO statuses, or verification
  results. Always call the appropriate tool and report exactly what it
  returns.
- If a verification fails (ISI mark not found, HUID invalid), tell the
  user clearly and suggest they report it as suspected counterfeit.
- For complex queries beyond your tools (e.g. detailed Form-I filling,
  licence application process), give a brief overview and recommend they
  visit bis.gov.in or contact a BIS regional office.
- Never end the call without first saying goodbye out loud.

When the session starts, greet the caller and invite them to ask their
question in any language.
"""


class GeminiLiveSession:
    def __init__(self, call_id: str, system_prompt: str):
        self.call_id = call_id
        self.system_prompt = system_prompt
        self.session = None
        self._context = None

    async def __aenter__(self):
        self._context = gemini_client.aio.live.connect(
            model=GEMINI_MODEL,
            config=types.LiveConnectConfig(
                response_modalities=["AUDIO"],
                system_instruction=self.system_prompt,
                tools=TOOLS,
                input_audio_transcription=types.AudioTranscriptionConfig(),
                output_audio_transcription=types.AudioTranscriptionConfig(),
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name="Aoede"
                        )
                    )
                ),
            ),
        )
        self.session = await self._context.__aenter__()

        return self

    async def trigger_greeting(self):
        """Send the opening greeting instruction once the telephony media stream is confirmed active."""
        await self.session.send_client_content(
            turns={
                "role": "user",
                "parts": [{
                    "text": (
                        "The caller has just answered the phone and the media stream is active. Greet them "
                        "warmly in Hindi first and then in English as "
                        "instructed, then invite them to ask their question "
                        "in any language."
                    )
                }],
            },
            turn_complete=True,
        )

    async def __aexit__(self, exc_type, exc, tb):
        return await self._context.__aexit__(exc_type, exc, tb)

    async def send_audio(self, pcm16k: bytes):
        await self.session.send_realtime_input(
            audio=types.Blob(data=pcm16k, mime_type="audio/pcm;rate=16000")
        )

    async def send_tool_response(self, function_responses):
        await self.session.send_tool_response(
            function_responses=function_responses
        )


print("Gemini Live session class + BISVEDA tools ready.")


In [ ]:
# 8. Tool dispatch — executes the function Gemini asked for
#
# Each tool call is wrapped in try/except so one failure never crashes
# the bridge. Results are returned to Gemini as structured dicts.
# Every invocation is also logged to the query_log table.

def execute_tool_call(call_id: str, name: str, args: dict) -> dict:
    state = CALLS.setdefault(call_id, {
        "phone": None, "language": None,
        "stage": "greeting", "should_end": False,
    })

    try:
        if name == "search_standards":
            query = args.get("query", "")
            results = backend_search_standards(query)
            log_query(
                call_id, query, "search_standards", name,
                args, results, f"{len(results)} standards found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"search_standards -> {len(results)} results", call_id)
            return {"status": "ok", "results": results}

        elif name == "verify_isi_mark":
            cm_l = args.get("cm_l_number", "")
            result = backend_verify_isi_mark(cm_l)
            log_query(
                call_id, f"Verify ISI: {cm_l}", "verify_isi_mark", name,
                args, result,
                "Valid" if result.get("valid") else "Invalid/Not found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"verify_isi_mark({cm_l}) -> valid={result.get('valid')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "verify_huid":
            huid = args.get("huid", "")
            result = backend_verify_huid(huid)
            log_query(
                call_id, f"Verify HUID: {huid}", "verify_huid", name,
                args, result,
                "Valid" if result.get("valid") else "Invalid/Not found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"verify_huid({huid}) -> valid={result.get('valid')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "check_qco_compliance":
            product = args.get("product_name", "")
            result = backend_check_qco(product)
            log_query(
                call_id, f"QCO check: {product}", "check_qco", name,
                args, result,
                "Under QCO" if result.get("covered") else "Not under QCO",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"check_qco({product}) -> covered={result.get('covered')}", call_id)
            return {"status": "ok", "result": result}

        elif name == "find_testing_labs":
            product = args.get("product_or_standard", "")
            location = args.get("location")
            results = backend_find_labs(product, location)
            log_query(
                call_id, f"Find labs: {product} near {location}",
                "find_labs", name, args, results,
                f"{len(results)} labs found",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"find_testing_labs -> {len(results)} labs", call_id)
            return {"status": "ok", "labs": results}

        elif name == "get_standard_passport":
            std_num = args.get("standard_number", "")
            # Normalize: "IS1239" -> "IS 1239"
            clean = std_num.strip().upper()
            if clean.startswith("IS") and len(clean) > 2 and clean[2].isdigit():
                clean = "IS " + clean[2:]
            result = MOCK_STANDARDS.get(clean, {
                "error": f"Standard '{std_num}' not found in database"
            })
            log_query(
                call_id, f"Standard passport: {std_num}",
                "standard_passport", name, args, result,
                result.get("title", "Not found"),
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"get_standard_passport({clean})", call_id)
            return {"status": "ok", "standard": result}

        elif name == "report_counterfeit":
            report = {
                "report_id": f"CF-{call_id[:8]}-{int(time.time()) % 10000}",
                "product": args.get("product_description"),
                "location": args.get("location"),
                "suspicious_number": args.get("suspicious_number"),
                "details": args.get("additional_details"),
                "status": "logged_for_investigation",
            }
            log_query(
                call_id, f"Counterfeit report: {args.get('product_description')}",
                "report_counterfeit", name, args, report,
                f"Report {report['report_id']} filed",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"report_counterfeit -> {report['report_id']}", call_id)
            return {"status": "ok", "report": report}

        elif name == "file_complaint":
            complaint = {
                "complaint_id": f"CMP-{call_id[:8]}-{int(time.time()) % 10000}",
                "product": args.get("product_description"),
                "details": args.get("complaint_details"),
                "cm_l": args.get("cm_l_number"),
                "purchase_location": args.get("purchase_location"),
                "status": "registered_pending_review",
            }
            log_query(
                call_id, f"Complaint: {args.get('product_description')}",
                "file_complaint", name, args, complaint,
                f"Complaint {complaint['complaint_id']} filed",
                state.get("language"),
            )
            logger.info("TOOL_RESULT", f"file_complaint -> {complaint['complaint_id']}", call_id)
            return {"status": "ok", "complaint": complaint}

        elif name == "end_call":
            state["should_end"] = True
            reason = args.get("reason", "completed")
            state["end_reason"] = reason
            update_call_session(call_id, end_reason=reason)
            logger.info("CALL_ENDING", f"Agent requested end (reason={reason})", call_id)
            return {"status": "ok"}

        else:
            logger.warn("TOOL_ERROR", f"Unknown tool: {name}", call_id)
            return {"status": "error", "error": f"unknown tool '{name}'"}

    except Exception as e:
        logger.error("TOOL_ERROR", f"Tool '{name}' failed: {type(e).__name__}: {e}", call_id)
        logger.error("TOOL_ERROR_TRACE", traceback.format_exc(), call_id)
        return {"status": "error", "error": f"{type(e).__name__}: {e}"}


print("BISVEDA tool dispatcher ready.")


In [ ]:
# 9. Exotel AgentStream WebSocket bridge — Continuous Multi-turn Gemini Live
#
# Connects Exotel 8kHz PCM16 bidirectional audio stream directly to Gemini Live
# (gemini-3.1-flash-live-preview) with automatic greeting and continuous multi-turn
# listening across the entire phone call.

app = FastAPI(title="BISVEDA Exotel Voice Bridge")
ACTIVE_CALLS = {}

@app.get("/")
def root():
    return {
        "service": "BISVEDA AI Voice Assistant",
        "transport": "Exotel AgentStream",
        "websocket": "/bisveda/media",
        "audio": "PCM16 mono 8000Hz",
        "model": GEMINI_MODEL,
        "active_calls": len(ACTIVE_CALLS),
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "BISVEDA AI Voice Assistant",
        "transport": "Exotel AgentStream",
        "account_sid": EXOTEL_ACCOUNT_SID,
        "caller_id": EXOTEL_CALLER_ID,
        "stream_url": globals().get("EXOTEL_STREAM_BASE", ""),
        "active_calls": len(ACTIVE_CALLS),
    }

async def _read_until_start(websocket: WebSocket):
    while True:
        raw = await websocket.receive_text()
        try:
            event = json.loads(raw)
        except json.JSONDecodeError:
            continue
        event_type = event.get("event")
        if event_type == "connected":
            continue
        if event_type == "start":
            start = event.get("start") or {}
            stream_sid = event.get("stream_sid") or start.get("stream_sid")
            call_sid = start.get("call_sid") or stream_sid or f"call_{int(time.time())}"
            phone = start.get("from") or "unknown"
            return {"call_id": call_sid, "stream_sid": stream_sid, "phone": phone}
        if event_type == "stop":
            return None

@app.websocket("/bisveda/media")
async def bisveda_media(websocket: WebSocket):
    await websocket.accept()
    call_id = f"call_{int(time.time())}"
    started_at = time.time()

    try:
        start_info = await _read_until_start(websocket)
        if not start_info:
            await websocket.close()
            return

        call_id = start_info["call_id"]
        stream_sid = start_info["stream_sid"]
        phone = start_info["phone"]
        ACTIVE_CALLS[call_id] = {"stream_sid": stream_sid, "phone": phone, "started_at": started_at}
        logger.info("EXOTEL_START", f"stream_sid={stream_sid} | phone={phone}", call_id)

        async with gemini_client.aio.live.connect(
            model=GEMINI_MODEL,
            config=types.LiveConnectConfig(
                response_modalities=["AUDIO"],
                system_instruction=SYSTEM_PROMPT,
                speech_config=types.SpeechConfig(
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Aoede")
                    )
                ),
            ),
        ) as gemini:
            logger.info("GEMINI_CONNECTED", "Gemini Live session connected", call_id)

            async def exotel_to_gemini():
                in_pkts = 0
                try:
                    while True:
                        raw = await websocket.receive_text()
                        try:
                            event = json.loads(raw)
                        except json.JSONDecodeError:
                            continue
                        event_type = event.get("event")
                        if event_type == "media":
                            payload = (event.get("media") or {}).get("payload")
                            if payload:
                                in_pkts += 1
                                pcm16k = exotel_to_gemini_pcm16(payload, call_id)
                                await gemini.send_realtime_input(
                                    audio=types.Blob(data=pcm16k, mime_type="audio/pcm;rate=16000")
                                )
                        elif event_type == "stop":
                            return
                except Exception:
                    return

            async def gemini_to_exotel():
                out_pkts = 0
                turn = 0
                while True:
                    turn += 1
                    try:
                        async for response in gemini.receive():
                            sc = getattr(response, "server_content", None)
                            if sc and getattr(sc, "model_turn", None):
                                for part in sc.model_turn.parts:
                                    inline = getattr(part, "inline_data", None)
                                    if inline and inline.data:
                                        chunks = gemini_to_exotel_chunks(inline.data, call_id)
                                        for p in chunks:
                                            out_pkts += 1
                                            await websocket.send_text(json.dumps({
                                                "event": "media",
                                                "stream_sid": stream_sid,
                                                "media": {"payload": p},
                                            }))
                            if sc and getattr(sc, "turn_complete", False):
                                flush = gemini_to_exotel_chunks(b"", call_id, flush=True)
                                for p in flush:
                                    await websocket.send_text(json.dumps({
                                        "event": "media",
                                        "stream_sid": stream_sid,
                                        "media": {"payload": p},
                                    }))
                                logger.info("TURN_COMPLETE", f"Turn {turn} complete | pkts={out_pkts}", call_id)
                    except Exception as e:
                        msg = str(e).lower()
                        if "1000" in msg or "closed" in msg or isinstance(e, asyncio.CancelledError):
                            break
                        await asyncio.sleep(0.05)

            in_task = asyncio.create_task(exotel_to_gemini())
            out_task = asyncio.create_task(gemini_to_exotel())

            # Trigger opening greeting
            async def send_greeting():
                await asyncio.sleep(0.2)
                await gemini.send_client_content(
                    turns={
                        "role": "user",
                        "parts": [{
                            "text": (
                                "The caller has answered the call. Please speak your opening greeting out loud now: "
                                "Greet them warmly in Hindi first and then in English as BISVEDA AI Helpline, "
                                "and invite them to ask their question about Indian Standards, ISI marks, or gold hallmarking."
                            )
                        }],
                    },
                    turn_complete=True,
                )
                logger.info("GREETING_TRIGGERED", "Opening greeting dispatched", call_id)

            asyncio.create_task(send_greeting())

            done, pending = await asyncio.wait(
                {in_task, out_task},
                return_when=asyncio.FIRST_COMPLETED,
            )
            for t in pending:
                t.cancel()

    except Exception as e:
        logger.error("BRIDGE_ERROR", str(e), call_id)
    finally:
        ACTIVE_CALLS.pop(call_id, None)
        clear_audio_state(call_id)
        logger.info("CALL_FINISHED", f"Duration: {round(time.time() - started_at, 1)}s", call_id)

print("Exotel AgentStream route registered: /bisveda/media")


In [ ]:
# 10. Start FastAPI server

import socket
import uvicorn


def _port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(("0.0.0.0", port))
            return True
        except OSError:
            return False


def run_server(port: int):
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning")


if "server_thread" in globals() and server_thread.is_alive():
    print(f"FastAPI already running on port {PORT} (reusing thread).")
else:
    if not _port_is_free(PORT):
        original_port = PORT
        candidate = PORT
        while not _port_is_free(candidate) and candidate < original_port + 20:
            candidate += 1
        print(f"Port {original_port} in use — using {candidate}.")
        PORT = candidate

    server_thread = threading.Thread(
        target=run_server, args=(PORT,), daemon=True
    )
    server_thread.start()
    time.sleep(2)
    print(f"FastAPI running on port {PORT}")


In [ ]:
# 11. Start ngrok tunnel

from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)

try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

if public_url.startswith("https://"):
    public_wss_url = "wss://" + public_url[len("https://"):]
elif public_url.startswith("http://"):
    public_wss_url = "wss://" + public_url[len("http://"):]
else:
    public_wss_url = public_url.replace("http://", "wss://")

EXOTEL_STREAM_BASE = public_wss_url + "/bisveda/media"
EXOTEL_WSS_URL     = EXOTEL_STREAM_BASE

print("=" * 70)
print("NGROK READY")
print("=" * 70)
print("Public HTTPS  :", public_url)
print("Exotel WSS    :", EXOTEL_STREAM_BASE)
print("=" * 70)


In [ ]:
# 12. Place an outbound call (for demo / testing)
#
# call_bisveda(phone_number) places an Exotel outbound call to your phone.
# When you answer, you immediately hear the BISVEDA greeting and can
# ask questions in Hindi, English, or any Indian language.

import requests

def call_bisveda(phone_number: str = None):
    """Place an outbound BISVEDA call via Exotel AgentStream."""
    phone = phone_number or MY_PHONE_NUMBER
    digits = "".join(filter(str.isdigit, str(phone)))
    if len(digits) == 12 and digits.startswith("91"):
        digits = digits[2:]
    elif len(digits) == 11 and digits.startswith("0"):
        digits = digits[1:]

    stream_url = f"{EXOTEL_STREAM_BASE}?sample-rate={EXOTEL_SAMPLE_RATE}"
    url = f"https://api.exotel.com/v1/Accounts/{EXOTEL_ACCOUNT_SID}/Calls/connect.json"

    payload = {
        "From": digits,
        "CallerId": EXOTEL_CALLER_ID,
        "StreamUrl": stream_url,
        "StreamType": "bidirectional",
    }

    print(f"Calling {digits} via Exotel...")
    response = requests.post(
        url,
        auth=(EXOTEL_API_KEY, EXOTEL_API_TOKEN),
        data=payload,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    call_sid = data.get("Call", {}).get("Sid")
    print(f"Call placed successfully! Call SID: {call_sid}")
    print(f"Answer your phone ({digits}) to speak with BISVEDA AI.")
    return data

print("call_bisveda() ready — run call_bisveda(MY_PHONE_NUMBER) to place a live test call.")


In [ ]:
# 13. Place the live demo call
#
# Run this cell to call your phone. When you pick up:
#   - Listen to the opening greeting
#   - Ask: "Mera ISI mark check karna hai — number hai CM/L-1234567"
#   - Ask: "Steel pipes ke liye kaunsa standard lagta hai?"
#   - Ask: "Is my gold hallmark real? HUID is AB1234CD"
#   - Ask: "Kya cement QCO ke under aata hai?"

call_bisveda(MY_PHONE_NUMBER)


In [ ]:
# 14. View call sessions and query log (run after a call ends)

import pandas as pd
from IPython.display import display, HTML

sessions = fetch_all_sessions()
queries = fetch_all_queries()

if sessions:
    print(f"\n{'='*60}")
    print(f"  CALL SESSIONS ({len(sessions)})")
    print(f"{'='*60}")
    df_sessions = pd.DataFrame(sessions)
    cols = [
        "call_id", "phone_number", "detected_language",
        "total_queries", "duration_seconds", "status",
    ]
    display(df_sessions[[c for c in cols if c in df_sessions.columns]])
else:
    print("No call sessions yet.")

if queries:
    print(f"\n{'='*60}")
    print(f"  QUERY LOG ({len(queries)})")
    print(f"{'='*60}")
    df_queries = pd.DataFrame(queries)
    cols = [
        "call_id", "timestamp", "detected_intent",
        "tool_called", "response_summary", "language",
    ]
    display(df_queries[[c for c in cols if c in df_queries.columns]])
else:
    print("No queries logged yet.")


In [ ]:
# 15. View and download the call log

from IPython.display import display, FileLink

with open(LOG_FILE, "r", encoding="utf-8") as f:
    log_text = f.read()

print(log_text[-3000:] if len(log_text) > 3000 else log_text)
display(FileLink(LOG_FILE, result_html_prefix="Download text log: "))
display(FileLink(LOG_JSONL_FILE, result_html_prefix="Download JSONL log: "))


In [ ]:
# 16. Analytics — query distribution, language breakdown, tool usage
#
# Run after several calls to see patterns.

import pandas as pd
from IPython.display import display

queries = fetch_all_queries()
sessions = fetch_all_sessions()

if not queries:
    print("No queries yet — run some test calls first.")
else:
    df = pd.DataFrame(queries)

    print("\n" + "=" * 50)
    print("  QUERY DISTRIBUTION BY INTENT")
    print("=" * 50)
    if "detected_intent" in df.columns:
        display(df["detected_intent"].value_counts().to_frame("count"))

    print("\n" + "=" * 50)
    print("  TOOL USAGE")
    print("=" * 50)
    if "tool_called" in df.columns:
        display(df["tool_called"].value_counts().to_frame("count"))

    if sessions:
        df_s = pd.DataFrame(sessions)
        print("\n" + "=" * 50)
        print("  LANGUAGE BREAKDOWN")
        print("=" * 50)
        if "detected_language" in df_s.columns:
            display(
                df_s["detected_language"].value_counts().to_frame("calls")
            )

        print("\n" + "=" * 50)
        print("  CALL OUTCOMES")
        print("=" * 50)
        if "status" in df_s.columns:
            display(df_s["status"].value_counts().to_frame("count"))

        if "duration_seconds" in df_s.columns:
            durations = pd.to_numeric(df_s["duration_seconds"], errors="coerce")
            print(f"\nAvg call duration: {durations.mean():.1f}s")
            print(f"Avg queries/call:  {df_s['total_queries'].astype(float).mean():.1f}")


## How to test end-to-end

### 1. Start the notebook
Run cells **1 → 13** once per fresh Kaggle runtime.

### 2. Configure Exotel
Use an **inbound App Bazaar flow** with a **Voicebot applet** and point the
Voicebot's AgentStream/WebSocket destination at the `EXOTEL_WSS_URL` printed
by cell 11. Associate your ExoPhone with that flow.

This matches Exotel's current inbound flow model:

```text
Caller → ExoPhone → App Bazaar Flow → Voicebot Applet → AgentStream WSS
```

### 3. Call from your phone
Call the Exotel number. You should hear:

> Namaste! BISVEDA mein aapka swagat hai. Welcome to BISVEDA.

Then try:

| Language | Sample query |
|----------|-------------|
| Hindi | "Mera ISI mark check karna hai — number hai CM/L-1234567" |
| Hindi | "Steel pipes ke liye kaunsa standard lagta hai?" |
| Hindi | "Kya cement QCO ke under aata hai?" |
| Hindi | "Mumbai mein testing lab batao" |
| English | "Is my gold hallmark genuine? The HUID is AB1234CD" |
| English | "What is IS 10500?" |
| English | "Do I need BIS certification for electronic toys?" |
| Hinglish | "Mujhe IS 1239 ke baare mein batao" |

### 4. What the assistant does
1. Detects the caller's language from speech.
2. Keeps the conversation in that language.
3. Calls the appropriate BISVEDA tool.
4. Speaks the tool result back to the caller.
5. Supports multi-turn follow-ups.

### Troubleshooting checklist

**You hear nothing / BISVEDA does not greet**
- Check the notebook for `EXOTEL_START`.
- Check that the Voicebot Applet points to the exact current `EXOTEL_WSS_URL`.
- Confirm the notebook cell 10 server is still alive.
- Open the printed `/health` URL in a browser.

**You see `connected` but no `EXOTEL_START`**
- The Voicebot/WebSocket negotiation reached your server, but Exotel has
  not completed the stream start. Check the Voicebot applet/flow.

**You see `EXOTEL_START`, `GEMINI_CONNECTED`, but no `CALLER_AUDIO_START`**
- The WebSocket is up but Exotel is not delivering media. Check the Voicebot
  applet mode and the stream URL/sample-rate configuration.

**You see `CALLER_AUDIO_START` but BISVEDA cannot hear you**
- Check Gemini API/key/model logs. The notebook expects raw PCM16 from Exotel,
  not μ-law.

**You hear BISVEDA but audio is clipped/laggy**
- Keep Exotel at 8 kHz for this prototype.
- Keep the current 3200-byte outgoing chunking.
- Do not reintroduce μ-law conversion from the old Twilio version.

### Production notes

| Concern | Demo | Production |
|------------|------------|------------|
| ASR/TTS | Gemini Live native audio | Bhashini ASR/TTS + local fallback |
| Standards DB | Mock dict | BISVEDA hybrid RAG / knowledge graph |
| ISI verification | Mock lookup | Real BIS verification service |
| HUID verification | Mock lookup | Real BIS HUID service |
| QCO database | Mock keyword match | Current QCO dataset + notifications |
| Lab directory | Mock list | Full BIS lab directory with geo-filtering |
| Telephony | Exotel inbound Voicebot | Exotel / other carrier as required |
| Database | SQLite | PostgreSQL |
| Tunnel | ngrok | Permanent public WSS / cloud deployment |
| Languages | Gemini native languages | Bhashini / 22 scheduled languages |
